### SourceLoader（文档加载器）

RAG 的第一步是**加载数据源**。LangChain 把所有来源统一成 `Document`：

- `page_content`：正文文本；
- `metadata`：来源、页码等结构化信息（检索/引用时会用到）。

加载器都实现 `BaseLoader` 接口：

- `.load()`：一次性返回 `list[Document]`；
- `.lazy_load()`：返回生成器，逐条产出（适合大数据集）。

下面先准备一批示例文件（txt / md / csv / json / pdf），再逐个演示常用加载器。

> 说明：这些加载器来自 `langchain-community` 包（当前已进入维护期，未来建议迁移到对应的独立包）。

In [ ]:
import json
import warnings
from pathlib import Path

# WebBaseLoader 会读取 USER_AGENT 标识；langchain-community 已进入维护期
warnings.filterwarnings("ignore", message=".*langchain-community is being sunset.*")
import os

os.environ.setdefault("USER_AGENT", "langchain-demo/1.0")

from langchain_community.document_loaders import (
    CSVLoader,
    DirectoryLoader,
    JSONLoader,
    PyPDFLoader,
    TextLoader,
    WebBaseLoader,
)
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document

DATA_DIR = Path("temp/source_loader_demo")
DATA_DIR.mkdir(parents=True, exist_ok=True)

(DATA_DIR / "hello.txt").write_text(
    "LangChain 是一个用于构建 LLM 应用的框架。\n它支持文档加载、切分、检索。",
    encoding="utf-8",
)
(DATA_DIR / "notes.md").write_text(
    "# 标题\n\n这是 Markdown 文档。\n\n- 要点一\n- 要点二\n", encoding="utf-8"
)
(DATA_DIR / "users.csv").write_text("name,role\n小明,后端\n小红,前端\n", encoding="utf-8")
(DATA_DIR / "items.json").write_text(
    json.dumps(
        [
            {"text": "Python 适合后端", "source": "a"},
            {"text": "Vue 适合前端", "source": "b"},
        ],
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


def make_pdf(path: Path, text: str) -> None:
    """生成一页含指定文字的极简 PDF（仅用于演示，无需额外依赖）。"""
    objs = [
        "<< /Type /Catalog /Pages 2 0 R >>",
        "<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
        "<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Contents 4 0 R "
        "/Resources << /Font << /F1 5 0 R >> >> >>",
    ]
    stream = f"BT /F1 24 Tf 72 700 Td ({text}) Tj ET"
    objs.append(f"<< /Length {len(stream)} >>\nstream\n{stream}\nendstream")
    objs.append("<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>")

    out = "%PDF-1.4\n"
    offsets = []
    for i, obj in enumerate(objs, start=1):
        offsets.append(len(out))
        out += f"{i} 0 obj\n{obj}\nendobj\n"
    xref_pos = len(out)
    out += f"xref\n0 {len(objs) + 1}\n0000000000 65535 f \n"
    for offset in offsets:
        out += f"{offset:010d} 00000 n \n"
    out += f"trailer\n<< /Size {len(objs) + 1} /Root 1 0 R >>\nstartxref\n{xref_pos}\n%%EOF\n"
    path.write_bytes(out.encode("latin-1"))


make_pdf(DATA_DIR / "doc.pdf", "Hello RAG PDF")


def show(tag: str, docs: list[Document]) -> None:
    print(f"[{tag}] {len(docs)} 个 Document")
    for doc in docs:
        print("  page_content:", repr(doc.page_content[:70]))
        print("  metadata:", doc.metadata)


print("示例文件已生成于：", DATA_DIR.resolve())


#### 1. TextLoader：加载纯文本文件

In [ ]:
docs = TextLoader(str(DATA_DIR / "hello.txt"), encoding="utf-8").load()
show("TextLoader", docs)

#### 2. DirectoryLoader：批量加载整个目录

In [ ]:
# 递归加载目录下所有 .txt（glob 控制匹配）
docs = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
).load()
show("DirectoryLoader", docs)

# 也支持懒加载（生成器，逐条产出，适合大数据集）
for doc in DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
).lazy_load():
    print("lazy_load ->", doc.metadata["source"])

#### 3. CSVLoader：逐行加载 CSV

In [ ]:
# CSV：每一行变成一个 Document，page_content 是「列名: 值」，metadata 带 row
docs = CSVLoader(str(DATA_DIR / "users.csv"), encoding="utf-8").load()
show("CSVLoader", docs)

#### 4. JSONLoader：按 jq 表达式抽取 JSON 内容

In [ ]:
# JSON：用 jq_schema 指定要抽取的内容（这里是每个元素的 text 字段）
docs = JSONLoader(str(DATA_DIR / "items.json"), jq_schema=".[].text").load()
show("JSONLoader", docs)

#### 5. PyPDFLoader：按页加载 PDF

In [ ]:
# PDF：按页加载，metadata 含 page / total_pages 等
docs = PyPDFLoader(str(DATA_DIR / "doc.pdf")).load()
print("页数:", len(docs))
print("内容:", docs[0].page_content)
print("metadata:", docs[0].metadata)

#### 6. WebBaseLoader：加载网页

In [ ]:
# 网页：抓取并把 HTML 解析为纯文本（需要联网）
docs = WebBaseLoader("https://github.com/pydantic/pydantic-ai").load()
print("来源:", docs[0].metadata["source"])
print("标题:", docs[0].metadata.get("title"))
print("正文片段:", docs[0].page_content)

#### 7. 自定义 Loader

内置加载器覆盖不到时，只要继承 `BaseLoader` 并实现 `lazy_load` 即可。（例如 `UnstructuredMarkdownLoader` 需要额外的 `unstructured` 依赖，这里用自定义 Loader 替代。）

In [8]:
# 自定义 Loader：实现 lazy_load 即可（BaseLoader 会据此提供 load / aload）
class MarkdownLoader(BaseLoader):
    """把 Markdown 文件整体加载为一个 Document。"""

    def __init__(self, file_path: str) -> None:
        self.file_path = file_path

    def lazy_load(self):
        text = Path(self.file_path).read_text(encoding="utf-8")
        yield Document(page_content=text, metadata={"source": self.file_path})


docs = list(MarkdownLoader(str(DATA_DIR / "notes.md")).lazy_load())
show("自定义 MarkdownLoader", docs)

[自定义 MarkdownLoader] 1 个 Document
  page_content: '# 标题\n\n这是 Markdown 文档。\n\n- 要点一\n- 要点二\n'
  metadata: {'source': 'temp/source_loader_demo/notes.md'}


#### 小结

| 加载器 | 来源 | 特点 |
| --- | --- | --- |
| `TextLoader` | 单个文本文件 | 最简单 |
| `DirectoryLoader` | 目录 | 用 `glob` 递归匹配，配合 `loader_cls` |
| `CSVLoader` | CSV | 每行一个 Document |
| `JSONLoader` | JSON | 用 `jq_schema` 抽取，需 `jq` |
| `PyPDFLoader` | PDF | 按页加载，metadata 含页码 |
| `WebBaseLoader` | 网页 | 抓取并解析 HTML |

**要点**

1. 一切加载结果都是 `Document(page_content, metadata)`；
2. `load()` 返回列表，`lazy_load()` 返回生成器；
3. 加载器的 `metadata` 会带上 `source`，是后续做引用溯源的依据；
4. 找不到现成加载器时，继承 `BaseLoader` 实现 `lazy_load` 即可。